# FlyRank Capstone — Refresh / Content Opportunity Scoring

**Functional, schema-adaptive version**

This notebook uses the real FlyRank Internship Warehouse through DuckDB. It does not assume the starter 30k-row column names: it detects the warehouse schema first, including the pseudonymized `client_hash_id` and `content_hash_id` keys.

It creates a forward-decline label, leakage-safe features, a transparent baseline, a Random Forest model, client-grouped validation, charts, a ranked review queue, metrics, and a GitHub Pages-ready research paper.

The raw warehouse and Hugging Face token are never saved to the repository.


## 1. Install and connect

The FlyRank dataset is gated. Accept the terms on Hugging Face and create a **READ** token first. The official dataset card recommends DuckDB over `hf://` for the full warehouse. citeturn0search0


In [ ]:
!pip -q install duckdb pandas numpy scikit-learn matplotlib pyarrow huggingface_hub

import os
import getpass
import json
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

SEED = 42

HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ").strip()
if not HF_TOKEN:
    raise ValueError("No token entered.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY = REL + "/fact_content_daily_performance/**/*.parquet"

print("DuckDB connection ready.")


## 2. Detect the actual warehouse schema

In [ ]:
schema = con.sql(
    "DESCRIBE SELECT * FROM read_parquet('" + DAILY + "')"
).df()

display(schema)
COLS = schema["column_name"].tolist()
print("Number of columns:", len(COLS))


In [ ]:
def choose(*names):
    for name in names:
        if name in COLS:
            return name
    return None

CLIENT = choose("client_hash_id", "client_id")
CONTENT = choose("content_hash_id", "content_id")
DATE = choose("report_date", "date", "day")

IMP = choose("impressions", "impressions_90d")
CLICKS = choose("clicks", "clicks_90d")
SESSIONS = choose("sessions", "sessions_90d")
PAGEVIEWS = choose("pageviews", "page_views")
USERS = choose("users")
ENGAGED = choose("engaged_sessions")
SCROLLS = choose("scroll_events")
AI_SESSIONS = choose("ai_sessions")

required = {
    "CLIENT": CLIENT,
    "CONTENT": CONTENT,
    "DATE": DATE,
    "IMP": IMP,
}

missing = [k for k,v in required.items() if v is None]
if missing:
    raise RuntimeError(
        "Required fields were not detected: " + str(missing) +
        ". Inspect the schema above."
    )

print("Detected column mapping:")
for key, value in {
    "CLIENT": CLIENT,
    "CONTENT": CONTENT,
    "DATE": DATE,
    "IMP": IMP,
    "CLICKS": CLICKS,
    "SESSIONS": SESSIONS,
    "PAGEVIEWS": PAGEVIEWS,
    "USERS": USERS,
    "ENGAGED": ENGAGED,
    "SCROLLS": SCROLLS,
    "AI_SESSIONS": AI_SESSIONS,
}.items():
    print(f"{key:14} -> {value}")


## 3. Find complete months

The notebook does not hard-code a month. It finds months with at least 28 distinct dates and uses the latest complete month as the future evaluation window and the preceding complete month as the feature window.


In [ ]:
months = con.sql(
    "SELECT DATE_TRUNC('month', CAST(" + DATE + " AS DATE)) AS month, "
    "COUNT(DISTINCT CAST(" + DATE + " AS DATE)) AS n_days "
    "FROM read_parquet('" + DAILY + "') "
    "GROUP BY 1 "
    "HAVING COUNT(DISTINCT CAST(" + DATE + " AS DATE)) >= 28 "
    "ORDER BY 1"
).df()

display(months.tail(12))

if len(months) < 2:
    raise RuntimeError("The warehouse needs at least two complete months.")

future_month = pd.Timestamp(months.iloc[-1]["month"])
feature_month = pd.Timestamp(months.iloc[-2]["month"])
previous_month = feature_month - pd.DateOffset(months=1)

future_start = future_month
future_end = future_month + pd.offsets.MonthEnd(0)
feature_start = feature_month
feature_end = feature_month + pd.offsets.MonthEnd(0)
previous_start = previous_month
previous_end = previous_month + pd.offsets.MonthEnd(0)
history_start = previous_start

print("Previous month:", previous_month.date())
print("Feature month :", feature_month.date())
print("Future month  :", future_month.date())


## 4. Build pre-future features and the forward label

In [ ]:
def metric_sum(column, start, end, alias):
    if column is None:
        return "0.0 AS " + alias + ","
    return (
        "SUM(CASE WHEN CAST(" + DATE + " AS DATE) BETWEEN DATE '"
        + str(start.date()) + "' AND DATE '" + str(end.date()) + "' "
        + "THEN COALESCE(" + column + ",0) ELSE 0 END) AS " + alias + ","
    )

select_parts = [
    metric_sum(IMP, previous_start, previous_end, "impressions_prev"),
    metric_sum(IMP, feature_start, feature_end, "impressions_feature"),
    metric_sum(CLICKS, feature_start, feature_end, "clicks_feature"),
    metric_sum(SESSIONS, feature_start, feature_end, "sessions_feature"),
    metric_sum(PAGEVIEWS, feature_start, feature_end, "pageviews_feature"),
    metric_sum(USERS, feature_start, feature_end, "users_feature"),
    metric_sum(ENGAGED, feature_start, feature_end, "engaged_feature"),
    metric_sum(SCROLLS, feature_start, feature_end, "scrolls_feature"),
    metric_sum(AI_SESSIONS, feature_start, feature_end, "ai_feature"),
    metric_sum(IMP, history_start, feature_end, "impressions_history"),
]

select_sql = "\n".join(select_parts).rstrip(",")
feature_sql = (
    "SELECT " + CLIENT + " AS client_key, " +
    CONTENT + " AS content_key,\n" +
    select_sql + "\n"
    "FROM read_parquet('" + DAILY + "')\n"
    "WHERE CAST(" + DATE + " AS DATE) BETWEEN DATE '" +
    str(history_start.date()) + "' AND DATE '" +
    str(feature_end.date()) + "'\n"
    "GROUP BY 1,2"
)

features = con.sql(feature_sql).df()
features = features[features["impressions_feature"] > 0].copy()

print("Feature rows:", len(features))
display(features.head())


In [ ]:
future_sql = (
    "SELECT " + CLIENT + " AS client_key, " +
    CONTENT + " AS content_key, "
    "SUM(COALESCE(" + IMP + ",0)) AS impressions_future "
    "FROM read_parquet('" + DAILY + "') "
    "WHERE CAST(" + DATE + " AS DATE) BETWEEN DATE '" +
    str(future_start.date()) + "' AND DATE '" +
    str(future_end.date()) + "' "
    "GROUP BY 1,2"
)

future = con.sql(future_sql).df()

df = features.merge(
    future,
    on=["client_key", "content_key"],
    how="left"
)

df["impressions_future"] = df["impressions_future"].fillna(0)

df["future_change"] = (
    (df["impressions_future"] - df["impressions_feature"])
    / df["impressions_feature"].clip(lower=1)
)

df["future_decline"] = (df["future_change"] <= -0.20).astype(int)

print("Eligible rows:", len(df))
print("Forward decline rate:", round(df["future_decline"].mean(), 4))


## 5. Leakage-safe model features

Only information available before the future month is used. The pseudonymous IDs are used for grouping/joining only and never enter the feature matrix.


In [ ]:
df["recent_change"] = (
    (df["impressions_feature"] - df["impressions_prev"])
    / df["impressions_prev"].clip(lower=1)
)

df["ctr_feature"] = (
    df["clicks_feature"] /
    df["impressions_feature"].clip(lower=1)
)

df["session_rate"] = (
    df["sessions_feature"] /
    df["impressions_feature"].clip(lower=1)
)

df["history_share"] = (
    df["impressions_feature"] /
    df["impressions_history"].clip(lower=1)
)

df["log_feature_impressions"] = np.log1p(df["impressions_feature"])
df["log_history_impressions"] = np.log1p(df["impressions_history"])

FEATURES = [
    "impressions_prev",
    "impressions_feature",
    "clicks_feature",
    "sessions_feature",
    "pageviews_feature",
    "users_feature",
    "engaged_feature",
    "scrolls_feature",
    "ai_feature",
    "impressions_history",
    "recent_change",
    "ctr_feature",
    "session_rate",
    "history_share",
    "log_feature_impressions",
    "log_history_impressions",
]

X = df[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["future_decline"].astype(int)

print("Features used:")
for f in FEATURES:
    print(" -", f)


## 6. Client-grouped validation

In [ ]:
if df["client_key"].nunique() < 2:
    raise RuntimeError("At least two clients are needed for validation.")

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["client_key"])
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Train rows:", len(X_train))
print("Test rows :", len(X_test))
print("Train clients:", df.iloc[train_idx]["client_key"].nunique())
print("Test clients :", df.iloc[test_idx]["client_key"].nunique())
print("Test decline rate:", round(y_test.mean(), 4))


## 7. Baseline and Random Forest

In [ ]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    order = np.argsort(-scores)[:k]
    return float(y_true[order].mean())

baseline_all = (
    -df["recent_change"].clip(-1, 1)
    * np.log1p(df["impressions_feature"])
)

baseline_test = baseline_all.iloc[test_idx].to_numpy()

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

rf.fit(X_train, y_train)

rf_test = rf.predict_proba(X_test)[:, 1]

K = min(50, len(y_test))

baseline_precision = precision_at_k(y_test, baseline_test, K)
rf_precision = precision_at_k(y_test, rf_test, K)
roc_auc = roc_auc_score(y_test, rf_test)
average_precision = average_precision_score(y_test, rf_test)

metrics = {
    "release": "FlyRank Internship Warehouse v20260703",
    "previous_month": str(previous_month.date()),
    "feature_month": str(feature_month.date()),
    "future_month": str(future_month.date()),
    "rows": int(len(df)),
    "clients": int(df["client_key"].nunique()),
    "test_rows": int(len(y_test)),
    "test_decline_rate": float(y_test.mean()),
    "baseline_precision_at_k": float(baseline_precision),
    "model_precision_at_k": float(rf_precision),
    "k": int(K),
    "roc_auc": float(roc_auc),
    "average_precision": float(average_precision),
    "precision_ratio_model_vs_baseline": (
        float(rf_precision / baseline_precision)
        if baseline_precision > 0 else None
    ),
    "validation": "20% client-grouped holdout",
    "label": "future-month impressions >= 20% below feature-month impressions",
    "seed": SEED,
}

print(json.dumps(metrics, indent=2))


## 8. Results charts

In [ ]:
FIG_DIR = Path("work/figures")
OUT_DIR = Path("work/outputs")
FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(7,5))
plt.bar(
    ["Baseline", "Random Forest"],
    [baseline_precision, rf_precision]
)
plt.ylabel("Precision@" + str(K))
plt.title("Model vs baseline")
plt.ylim(
    0,
    max(0.05, max(baseline_precision, rf_precision) * 1.25)
)
plt.tight_layout()
plt.savefig(
    FIG_DIR / "model_vs_baseline.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

importance = pd.Series(
    rf.feature_importances_,
    index=FEATURES
).sort_values()

plt.figure(figsize=(8,6))
importance.plot(kind="barh")
plt.xlabel("Feature importance")
plt.title("Random Forest feature importance")
plt.tight_layout()
plt.savefig(
    FIG_DIR / "feature_importance.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()


## 9. Ranked recommendations

In [ ]:
final_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

final_model.fit(X, y)
df["opportunity_score"] = final_model.predict_proba(X)[:, 1]

traffic_q75 = df["impressions_feature"].quantile(0.75)
ctr_median = df["ctr_feature"].median()

df["reason_recent_decline"] = df["recent_change"] <= -0.20
df["reason_high_traffic"] = df["impressions_feature"] >= traffic_q75
df["reason_low_ctr"] = df["ctr_feature"] < ctr_median
df["reason_model_priority"] = (
    df["opportunity_score"] >=
    df["opportunity_score"].quantile(0.90)
)

def choose_action(row):
    if (
        row["reason_model_priority"]
        and row["reason_recent_decline"]
        and row["reason_high_traffic"]
    ):
        return "HIGH_PRIORITY_REVIEW"
    if row["reason_recent_decline"]:
        return "REFRESH_REVIEW"
    if row["reason_low_ctr"] and row["reason_high_traffic"]:
        return "CTR_REVIEW"
    if row["reason_model_priority"]:
        return "MODEL_PRIORITY_REVIEW"
    return "MONITOR"

df["recommended_action"] = df.apply(
    choose_action,
    axis=1
)

action_summary = (
    df["recommended_action"]
    .value_counts()
    .rename_axis("action")
    .reset_index(name="items")
)

display(action_summary)


In [ ]:
# Local-only queue. Do not publish this file.
queue = df.sort_values(
    ["opportunity_score", "impressions_feature"],
    ascending=[False, False]
)

queue[[
    "content_key",
    "opportunity_score",
    "impressions_feature",
    "ctr_feature",
    "recent_change",
    "recommended_action"
]].head(500).to_csv(
    OUT_DIR / "ranked_content_actions_local.csv",
    index=False
)

metrics["recommendation_counts"] = {
    str(k): int(v)
    for k,v in df["recommended_action"].value_counts().items()
}

with open(OUT_DIR / "capstone_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved capstone_metrics.json")


## 10. Generate the research paper

In [ ]:
ratio_text = (
    f"{rf_precision / baseline_precision:.2f}x"
    if baseline_precision > 0 else "not defined"
)

paper = (
"# Refresh Opportunity Scoring: A Leakage-Safe Content Review Ranking System\n\n"
"## Abstract\n\n"
f"This study asks whether pre-future-window search-performance signals can rank content items for performance-risk review. "
f"Using the FlyRank Internship Warehouse v20260703, daily performance was aggregated through {feature_month.strftime('%B %Y')} "
f"and a forward decline label was created from {future_month.strftime('%B %Y')}. "
f"A transparent recent-decline baseline was compared with a Random Forest using a client-grouped holdout and explicit leakage controls. "
f"On the held-out test set, the baseline achieved Precision@{K} of **{baseline_precision:.3f}**, while the Random Forest achieved **{rf_precision:.3f}**, "
f"with ROC-AUC **{roc_auc:.3f}**. "
"These results support a decision-support ranking workflow, not a causal claim or a claim about Google's ranking algorithm.\n\n"
"## 1. Introduction / Problem Statement\n\n"
"Content teams cannot manually investigate every content item with equal depth. "
"The decision supported by this project is therefore prioritization: which items should receive human review first when observable signals indicate elevated future performance risk? "
"The output is a ranked review queue rather than an automatic publishing, pruning, or refresh system.\n\n"
"## 2. Data\n\n"
"The source is the FlyRank Internship Warehouse v20260703, a pseudonymized warehouse of daily content performance and related dimensions. "
f"The feature month is **{feature_month.strftime('%Y-%m')}** and the future evaluation month is **{future_month.strftime('%Y-%m')}**. "
f"The previous month ({previous_month.strftime('%Y-%m')}) is used to measure recent movement. "
"Items with zero feature-month impressions are excluded because percentage decline is undefined. "
"The query-level fixed 90-day table is not used because its fixed window is not aligned to the forward prediction cutoff. "
"No client names, domains, URLs, private queries, titles, credentials, or raw exports are included.\n\n"
"## 3. Methodology\n\n"
"### Label\n\n"
"An item is labeled as a forward decline when future-month impressions are at least 20% below feature-month impressions.\n\n"
"### Features\n\n"
"Features are computed only from data available before the future month. They include traffic volume, clicks, sessions, recent impression movement, CTR, session rate, historical traffic volume, and log-transformed traffic. "
"Pseudonymous IDs are used only for joins and grouping.\n\n"
"### Baseline\n\n"
"The baseline prioritizes negative recent impression movement weighted by traffic exposure.\n\n"
"### Model\n\n"
"A Random Forest classifier uses class balancing, constrained depth, minimum leaf size, and a fixed random seed.\n\n"
"### Validation\n\n"
"A 20% client-grouped holdout keeps content from the same client on only one side of the split.\n\n"
"## 4. Results\n\n"
"| Metric | Baseline | Random Forest |\n"
"|---|---:|---:|\n"
f"| Precision@{K} | {baseline_precision:.3f} | {rf_precision:.3f} |\n"
f"| ROC-AUC | — | {roc_auc:.3f} |\n"
f"| Average Precision | — | {average_precision:.3f} |\n\n"
f"Test-set decline base rate: **{y_test.mean():.3f}**. "
f"The model/baseline Precision@{K} ratio is **{ratio_text}**.\n\n"
"![Model vs baseline](../work/figures/model_vs_baseline.png)\n\n"
"![Feature importance](../work/figures/feature_importance.png)\n\n"
"## 5. Limitations & Honest Framing\n\n"
"This analysis is observational and directional. It identifies signals useful for ranking review priority under this validation design; it does not establish causality. "
"The 20% decline threshold is an operational choice. Performance can shift across time, clients, content types, and measurement systems. "
"The model should be revalidated on future windows before operational use.\n\n"
"## 6. Ranked Recommendations\n\n"
"1. **Review high-score items first.** Use the score to prioritize limited human review capacity.\n"
"2. **Investigate recent declines with meaningful traffic exposure.** These are strong candidates for content-quality or intent review.\n"
"3. **Treat low CTR as a separate review signal.** This can motivate metadata/SERP investigation but does not prove that changing metadata will improve traffic.\n"
"4. **Record a human decision.** Use refresh review, CTR review, model-priority review, or monitor as decision-support categories.\n"
"5. **Revalidate later.** Compare Precision@K with the same baseline when another future window becomes available.\n\n"
"## 7. Reproducibility\n\n"
"The complete analysis is in `work/notebooks/capstone.ipynb`. "
"The notebook queries the gated Hugging Face Parquet warehouse through DuckDB, discovers the real schema, uses explicit temporal windows, excludes identifiers from model features, performs client-grouped validation, and writes aggregate metrics to `work/outputs/capstone_metrics.json`. "
"The raw warehouse is not stored in the repository.\n\n"
"## 8. Acknowledgments & Data Credit\n\n"
"Built on the **FlyRank ML Internship dataset**.\n\n"
"[FlyRank](https://flyrank.ai)\n\n"
"## Public Safety\n\n"
"Only aggregate research results are intended for publication. Do not publish the local row-level queue, raw warehouse, credentials, client-identifying information, domains, URLs, private queries, or titles.\n"
)

Path("docs").mkdir(exist_ok=True)
Path("docs/index.md").write_text(paper, encoding="utf-8")
print("Generated docs/index.md")


## 11. Final checklist

- Run all cells successfully.
- Check `work/outputs/capstone_metrics.json`.
- Check both PNG charts.
- Push the notebook, paper, figures, and aggregate metrics to GitHub.
- Do **not** publish `ranked_content_actions_local.csv`.
- Do **not** publish the Hugging Face token.
- Enable GitHub Pages from `/docs`.
- Put the exact deployed URL in `submission/paper_url.txt`.
